# Task 5: PCA 压缩 ESM2，让结构特征浮出来

**目标**: 1280 维 ESM2 把 7~9 个结构特征完全淹没（重要性 top-30 全是 esm_*）。降维后看结构特征能否进入重要位次。

**关键约束**: PCA 必须在每折训练集内 fit，再 transform 验证集。
结构特征 (plddt/sasa/rsa/ss_*/delta_hydrophobicity) 不参与 PCA，原样拼接。

In [ ]:
import numpy as np, pandas as pd, warnings
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
warnings.filterwarnings("ignore")

BASE_PATH = "/mnt/volume6/czj/labLGN/LabLZ/xgboost_trial/"

# ===== 加载数据 =====
df_feat = pd.read_csv(BASE_PATH + "features_v3.csv")

ID_COLS = ["KEY", "Gene", "reloc_v3"]
NAN_PLACEHOLDERS = ["ddg", "plddt_diff"]
exclude_cols = ID_COLS + NAN_PLACEHOLDERS
feat_cols = [c for c in df_feat.columns if c not in exclude_cols]

# 识别 ESM2 列和结构列
esm_cols = [c for c in feat_cols if c.startswith("esm_")]
struct_cols_present = ["plddt", "sasa", "rsa", "ss_helix", "ss_strand",
                       "ss_coil", "delta_hydrophobicity", "struct_status"]

X_full = df_feat[feat_cols].values.astype(np.float32)
esm_idx = [feat_cols.index(c) for c in esm_cols]
struct_idx = [feat_cols.index(c) for c in struct_cols_present if c in feat_cols]
other_idx = [i for i in range(len(feat_cols)) if i not in esm_idx and i not in struct_idx]

X_esm = X_full[:, esm_idx]
X_struct = X_full[:, struct_idx]
X_other = X_full[:, other_idx] if other_idx else None

y_bin = (df_feat["reloc_v3"].values > 0).astype(int)
groups = df_feat["Gene"].values

print(f"ESM2 列: {len(esm_idx)}, 结构列: {len(struct_idx)}, 其他列: {len(other_idx)}")
print(f"n={len(y_bin)}, 正例={int(y_bin.sum())}, base_rate={y_bin.sum()/len(y_bin):.4f}")

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)


ESM2 列: 1280, 结构列: 8, 其他列: 0
n=2179, 正例=222, base_rate=0.1019


## 5a. PCA 维度扫描 (30 / 50 / 100)

In [2]:
pca_results = {}

for n_comp in [30, 50, 100]:
    print(f"\n{'='*50}")
    print(f"PCA n_components={n_comp}")
    print(f"{'='*50}")

    oof_pca = np.zeros(len(y_bin), dtype=np.float32)
    per_fold = []

    for fold, (tr_idx, te_idx) in enumerate(cv.split(X_full, y_bin, groups)):
        # --- ESM2: Impute → Scale → PCA (仅在训练集 fit) ---
        imp_e = SimpleImputer(strategy="median"); scl_e = StandardScaler()
        Xe_tr = scl_e.fit_transform(imp_e.fit_transform(X_esm[tr_idx])).astype(np.float32)
        Xe_te = scl_e.transform(imp_e.transform(X_esm[te_idx])).astype(np.float32)
        pca = PCA(n_components=n_comp, random_state=42)
        Xe_tr_pca = pca.fit_transform(Xe_tr).astype(np.float32)
        Xe_te_pca = pca.transform(Xe_te).astype(np.float32)

        # --- 结构特征: 只做 Impute + Scale ---
        imp_s = SimpleImputer(strategy="median"); scl_s = StandardScaler()
        Xs_tr = scl_s.fit_transform(imp_s.fit_transform(X_struct[tr_idx])).astype(np.float32)
        Xs_te = scl_s.transform(imp_s.transform(X_struct[te_idx])).astype(np.float32)

        # --- 拼接 ---
        X_tr_pca = np.hstack([Xe_tr_pca, Xs_tr])
        X_te_pca = np.hstack([Xe_te_pca, Xs_te])
        if X_other is not None:
            imp_o = SimpleImputer(strategy="median"); scl_o = StandardScaler()
            Xo_tr = scl_o.fit_transform(imp_o.fit_transform(X_other[tr_idx])).astype(np.float32)
            Xo_te = scl_o.transform(imp_o.transform(X_other[te_idx])).astype(np.float32)
            X_tr_pca = np.hstack([X_tr_pca, Xo_tr])
            X_te_pca = np.hstack([X_te_pca, Xo_te])

        # --- XGBoost ---
        y_tr = y_bin[tr_idx]; sw = compute_sample_weight("balanced", y_tr)
        model = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                              subsample=0.8, colsample_bytree=0.5,
                              objective="binary:logistic", eval_metric="aucpr",
                              random_state=42, n_jobs=-1, tree_method="hist")
        model.fit(X_tr_pca, y_tr, sample_weight=sw, verbose=False)
        oof_pca[te_idx] = model.predict_proba(X_te_pca)[:, 1]

        y_te = y_bin[te_idx]
        per_fold.append({
            "fold": fold,
            "auroc": roc_auc_score(y_te, oof_pca[te_idx]),
            "auprc": average_precision_score(y_te, oof_pca[te_idx]),
            "explained_var": pca.explained_variance_ratio_.sum()
        })

    auroc = roc_auc_score(y_bin, oof_pca)
    auprc = average_precision_score(y_bin, oof_pca)
    ev = np.mean([f["explained_var"] for f in per_fold])
    fa = [f["auroc"] for f in per_fold]
    fp = [f["auprc"] for f in per_fold]
    print(f"  AUROC={auroc:.4f} (per-fold: {[f'{v:.3f}' for v in fa]})")
    print(f"  AUPRC={auprc:.4f} (per-fold: {[f'{v:.3f}' for v in fp]})")
    print(f"  平均解释方差: {ev:.2f}")

    pca_results[n_comp] = {"auroc": auroc, "auprc": auprc, "explained_var": ev,
                            "oof": oof_pca, "per_fold": per_fold}

# ===== 维度扫描汇总 =====
print(f"\n{'='*60}")
print(f"  PCA 维度扫描汇总")
print(f"  {'PCA 维度':<15s} {'AUROC':>8s} {'AUPRC':>8s} {'解释方差':>10s}")
print(f"  {'Full (1280)':<15s} {'(见全量基线)':>8s}")
for nc in [30, 50, 100]:
    r = pca_results[nc]
    print(f"  {f'PCA({nc})':<15s} {r['auroc']:>8.4f} {r['auprc']:>8.4f} "
          f"{r['explained_var']:>10.2f}")
print(f"{'='*60}")



PCA n_components=30
  AUROC=0.6063 (per-fold: ['0.599', '0.645', '0.551', '0.625', '0.602'])
  AUPRC=0.1381 (per-fold: ['0.144', '0.170', '0.124', '0.162', '0.139'])
  平均解释方差: 0.49

PCA n_components=50
  AUROC=0.6087 (per-fold: ['0.639', '0.651', '0.528', '0.632', '0.582'])
  AUPRC=0.1388 (per-fold: ['0.133', '0.208', '0.110', '0.161', '0.144'])
  平均解释方差: 0.58

PCA n_components=100
  AUROC=0.5752 (per-fold: ['0.575', '0.626', '0.505', '0.593', '0.560'])
  AUPRC=0.1343 (per-fold: ['0.127', '0.193', '0.101', '0.162', '0.173'])
  平均解释方差: 0.70

  PCA 维度扫描汇总
  PCA 维度             AUROC    AUPRC       解释方差
  Full (1280)      (见全量基线)
  PCA(30)           0.6063   0.1381       0.49
  PCA(50)           0.6087   0.1388       0.58
  PCA(100)          0.5752   0.1343       0.70


## 5b. PCA(50) 后的特征重要性 —— 结构特征是否升上来了？

In [3]:
print("\n--- 特征重要性 PCA(50) (fit on all data 仅用于排名) ---")

# Fit on all data (仅用于画重要性，不用来算 AUROC)
imp_e = SimpleImputer(strategy="median"); scl_e = StandardScaler()
Xe_all = scl_e.fit_transform(imp_e.fit_transform(X_esm)).astype(np.float32)
pca_all = PCA(n_components=50, random_state=42)
Xe_pca_all = pca_all.fit_transform(Xe_all).astype(np.float32)

imp_s = SimpleImputer(strategy="median"); scl_s = StandardScaler()
Xs_all = scl_s.fit_transform(imp_s.fit_transform(X_struct)).astype(np.float32)

X_all_pca = np.hstack([Xe_pca_all, Xs_all]).astype(np.float32)

sw_ratio = (y_bin==0).sum() / max((y_bin==1).sum(), 1)
xgb_pca_fi = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                           subsample=0.8, colsample_bytree=0.5,
                           scale_pos_weight=sw_ratio,
                           objective="binary:logistic", eval_metric="aucpr",
                           random_state=42, n_jobs=-1, tree_method="hist")
xgb_pca_fi.fit(X_all_pca, y_bin, verbose=False)

pca_names = [f"PC{i+1}" for i in range(50)] + struct_cols_present
pca_imps = xgb_pca_fi.feature_importances_
idx_pca = np.argsort(pca_imps)[::-1]

print("Top-20 特征重要性 (PCA(50) 后):")
for rank, i in enumerate(idx_pca[:20]):
    val = pca_imps[i]; bar = "█" * int(val * 2000)
    print(f"  {rank+1:2d}. {pca_names[i]:30s}  {val:.5f}  {bar}")

print("\n结构特征在 PCA(50) 后的排名:")
for name in struct_cols_present:
    i = pca_names.index(name)
    rank = np.where(idx_pca == i)[0][0] + 1
    imp = pca_imps[i]
    marker = " ★ 进入 top-20!" if rank <= 20 else ""
    print(f"  {name:25s} 排名={rank:3d}/{len(pca_names)}  重要性={imp:.5f}{marker}")



--- 特征重要性 PCA(50) (fit on all data 仅用于排名) ---
Top-20 特征重要性 (PCA(50) 后):
   1. PC1                             0.03355  ███████████████████████████████████████████████████████████████████
   2. PC36                            0.02584  ███████████████████████████████████████████████████
   3. PC33                            0.02287  █████████████████████████████████████████████
   4. PC30                            0.02216  ████████████████████████████████████████████
   5. PC23                            0.02203  ████████████████████████████████████████████
   6. ss_helix                        0.02114  ██████████████████████████████████████████
   7. PC42                            0.02084  █████████████████████████████████████████
   8. PC13                            0.02066  █████████████████████████████████████████
   9. PC22                            0.02055  █████████████████████████████████████████
  10. PC26                            0.02053  ████████████████████████████████